# Knowledge / Belief Layer — Demo (issue #45)

Each character now carries a **knowledge** object: the facts it *believes* about
the world. Beliefs can be **incomplete** (the troll doesn't know a key exists) or
even **wrong** (a belief whose text contradicts the world graph). Some are known up
front; some are learned during play.

This is distinct from **memory** (#37): memory is the *episodic log* of what a
character saw or did; knowledge is its *current world-model* — "what's true now".

This notebook is fully **offline and deterministic** (no API key). It reuses the
real Action Castle cast via `build_game()` and seeds beliefs *in the notebook*,
leaving the canonical game pristine. It shows three things:

1. Two characters who know different things get different observations.
2. The **same** decision rule produces **different actions** because the knowledge
   differs.
3. A hidden item is **perceived only** by a character whose knowledge unlocks it.

In [1]:
from rich.console import Console

from hw1_solution.action_castle import build_game

# A rich console for colored notebook output. force_jupyter=True makes rich emit
# its HTML here (a notebook's stdout isn't a TTY, so it would otherwise go plain).
console = Console(force_jupyter=True, width=100)


def banner(title):
    """A colored section rule, matching the look of the engine's turn headers."""
    console.rule(title, style="bold cyan", align="left")


# Build the canonical Action Castle. Passing no llm_client keeps the scripted
# fallback behaviors; we won't run turns here — we inspect observations and drive
# decisions by hand — so the game is just a convenient pre-populated world.
game = build_game()

guard = game.characters["guard"]
troll = game.characters["troll"]
ghost = game.characters["ghost"]
princess = game.characters["princess"]

console.print("Characters:", list(game.characters.keys()))

Characters:
['The player', 'troll', 'guard', 'princess', 'ghost']

## 1. Seed asymmetric beliefs

We seed each NPC with what it *knows* — deliberately uneven. `add_belief(text,
topic=...)` records a prior (no turn stamp); a `topic` doubles as a key that can
unlock a hidden thing flagged with a matching `secret_topic` (see §4).

In [2]:
# The guard knows the tower is locked and that he holds the key (topic "door").
guard.add_belief(
    "The tower door is locked and I hold the only brass key.", topic="door"
)
guard.add_belief("The dungeon hides the castle's dark secret.")

# The troll's knowledge is INCOMPLETE: it knows only its bargain, nothing of a key.
troll.add_belief(
    "I am hungry; the guard promised to feed me for guarding the bridge."
)

# The ghost holds a secret nobody else does (topic "runes").
ghost.add_belief("The guard murdered me.")
ghost.add_belief("The runes on the strange candle can banish me.", topic="runes")

# The princess's knowledge is sparse and grief-tinted.
princess.add_belief("My beloved is gone; I weep alone in the tower.")

console.print("guard knows about 'door': ", guard.knowledge.knows_about("door"))
console.print("troll knows about 'door': ", troll.knowledge.knows_about("door"))
console.print("ghost knows about 'runes':", ghost.knowledge.knows_about("runes"))

guard knows about 'door':  True

troll knows about 'door':  False

ghost knows about 'runes': True

## 2. Different knowledge → different observations

`Game.describe_for(character)` now appends a **"What you know:"** section built from
that character's beliefs. Compare the guard's view to the troll's: same engine, same
world, different minds.

In [3]:
banner("GUARD sees:")
console.print(game.describe_for(guard))
console.print()
banner("TROLL sees:")
console.print(game.describe_for(troll))

GUARD sees: ────────────────────────────────────────────────────────────────────────────────────────

COURTYARD
You are in the courtyard of ACTION CASTLE.
Exits:
 * West to Drawbridge
 * Up to Tower Stairs
 * Down to Dungeon Stairs
 * East to Great Feasting Hall
Inventory:
 * key - a brass key 
 * sword - a short sword 
Available actions: adopt goal, attack, catch fish, describe, drink, drop, drop goal, eat, examine, 
get, ghost touch, give, go, growl, haunt, inventory, light, pick rose, pound fists, propose, quit, 
read runes, say, sequence, sit on throne, smell rose, snarl, take off, threaten, unlock door, 
unwield, wait, warn, wear, wear crown, wield
What you know:
 - The tower door is locked and I hold the only brass key.
 - The dungeon hides the castle's dark secret.
Turn: 0

TROLL sees: ────────────────────────────────────────────────────────────────────────────────────────

DRAWBRIDGE
You are standing on one side of a drawbridge leading to ACTION CASTLE.
Exits:
 * West to Winding Path
 * East to Courtyard
Inventory:
 * club - a heavy club 
Available actions: adopt goal, attack, catch fish, describe, drink, drop, drop goal, eat, examine, 
get, ghost touch, give, go, growl, haunt, inventory, light, pick rose, pound fists, propose, quit, 
read runes, say, sequence, sit on throne, smell rose, snarl, take off, threaten, unlock door, 
unwield, wait, warn, wear, wear crown, wield
What you know:
 - I am hungry; the guard promised to feed me for guarding the bridge.
Turn: 0

## 3. Same rule, different action

A belief is only useful if it changes behavior. Both NPCs below are driven by the
**same** `ScriptedAgent` rule — *"if I know about a brass key, try to unlock the
door; otherwise just look around."* The guard acts on the key; the troll, who never
learned it exists, cannot. Same rule, divergent action — driven purely by knowledge.

In [4]:
from text_adventure_games.npc import ScriptedAgent, build_npc_context


def key_aware_rule(observation):
    # Only a character whose observation mentions the brass key can plan to use it.
    if "brass key" in observation.lower():
        return "unlock door"
    return "look"


guard.set_agent(ScriptedAgent(key_aware_rule))
troll.set_agent(ScriptedAgent(key_aware_rule))

guard_cmd = guard.agent.decide(build_npc_context(guard, game))
troll_cmd = troll.agent.decide(build_npc_context(troll, game))

console.print("guard decides:", repr(guard_cmd))
console.print("troll decides:", repr(troll_cmd))

guard decides: 'unlock door'

troll decides: 'look'

## 4. Perception gating: hidden in plain sight

Knowledge doesn't just add a section — it can gate **perception**. Flag a thing with
`set_property("secret_topic", "<key>")` and `describe_for` reveals it only to a
character whose knowledge `knows_about("<key>")`. Below, a rune-etched candle sits in
an alcove; the ghost (who knows the runes) perceives it, the troll (who doesn't) does
not — same room, same candle.

In [5]:
from text_adventure_games import things

# Stage the ghost and the troll in one room so we can compare like-for-like.
alcove = things.Location("Hidden Alcove", "A quiet alcove tucked off the courtyard.")
for npc in (ghost, troll):
    npc.location.remove_character(npc)
    alcove.add_character(npc)

# A candle that only someone who knows the "runes" can even notice.
secret_candle = things.Item(
    "candle", "a strange candle", "A candle etched with glowing runes."
)
secret_candle.set_property("secret_topic", "runes")
alcove.add_item(secret_candle)

ghost_obs = game.describe_for(ghost)
troll_obs = game.describe_for(troll)

# Look only at the perception part (before the beliefs section), so we're testing
# what each character can SEE, not what the ghost happens to believe.
console.print("Is the candle among the items each one perceives?")
console.print("  ghost (knows 'runes'):", "candle" in ghost_obs.split("What you know:")[0])
console.print("  troll (does not):     ", "candle" in troll_obs.split("What you know:")[0])
console.print()
banner("Full GHOST observation")
console.print(ghost_obs)

Is the candle among the items each one perceives?

ghost (knows 'runes'): True

troll (does not):      False

Full GHOST observation ─────────────────────────────────────────────────────────────────────────────

HIDDEN ALCOVE
A quiet alcove tucked off the courtyard.
Items here:
 * candle - a strange candle 
Characters here:
 * troll - A mean troll
Inventory:
 * crown - a crown 
Available actions: adopt goal, attack, catch fish, describe, drink, drop, drop goal, eat, examine, 
get, ghost touch, give, go, growl, haunt, inventory, light, pick rose, pound fists, propose, quit, 
read runes, say, sequence, sit on throne, smell rose, snarl, take off, threaten, unlock door, 
unwield, wait, warn, wear, wear crown, wield
What you know:
 - The guard murdered me.
 - The runes on the strange candle can banish me.
Turn: 0

## How knowledge composes with memory (#37)

The two layers are deliberately separate and feed `describe_for` in **different**
sections:

| Layer | What it is | How it's filled | Observation section |
|-------|-----------|-----------------|---------------------|
| **Knowledge** (#45, this notebook) | the current world-model — possibly wrong | seeded up front or via explicit `learn()` | "What you know:" |
| **Memory** (#75, notebook `09`) | the episodic log of what was seen/done | auto-captured from outcomes + `Game.events` | "Relevant memories:" |

**Invariant:** beliefs are *context, not authority*. The world graph stays the single
source of truth — a belief may be false, but only actions through the precondition
gate ever change the world.